# $\log(3/2)$: historical CMF bound

Each section constructs a CMF, walks its trajectory, and prints $\delta_N$ at source depth 1000 for the reduced rational approximant. A finite-depth value does not prove the published measure bound.

In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

The CMFs are constructed directly by `ramanujantools.cmf.pfq.pFq`. The starting readouts, walks, and approximants are exact.

In [ ]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx

from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
with ctx.workdps(7000):
    TARGET = arb(3).log() - arb(2).log()
DEPTHS = list(range(10, 1001, 10))
x0, x1, y0 = sp.symbols("x0 x1 y0")
results = {}


def delta(approximant):
    """Finite-depth exponent of the reduced rational approximant."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2*q.bit_length() + 128):
        error = abs(TARGET - arb(p)/q)
        return float((-1 - error.log()/arb(q).log()).mid())

## 1. Alladi–Robinson (1980): $\log(3/2)$

[Integral construction and measure formula](https://www.math.ru.nl/~wzudilin/gat/oALLADI4.txt).
$$E_n=\int_0^1\frac{t^n(1-t)^n}{(t+2)^{n+1}}\,dt.$$
**CMF:** rank 2, `${}_2F_1(-1/2)` ? **Start:** $(2,2;4)$ ? **Direction:** $(1,1;2)$.
The readout is $E_n=C_n+B_n\log(3/2)$, $p_n/q_n=-C_n/B_n$. Depth $N$ means the integral index $n=N$.
The beta-integral normalization is common to both coefficient rows and cancels in the readout; the code uses the $(F,\theta F)$ basis.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 2))
start = {x0: 2, x1: 2, y0: 4}
trajectory = {x0: 1, x1: 1, y0: 2}
walks = cmf.walk(trajectory, [N - 1 for N in DEPTHS], start)
C = Matrix([[-6, 17]])
B = Matrix([[15, -42]])
approximants = [-(C*M)[0]/(B*M)[0] for M in walks]

deltas = [delta(r) for r in approximants]
results["Alladi–Robinson (1980)"] = (DEPTHS, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## All constructions

$\delta_N$ against the source depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published upper bound $\mu_0$ | $\delta_{\rm eq}$ | $\delta_{1000}$ | CMF |
|:--|--:|--:|--:|:--|
| Alladi–Robinson | 3.547470591 | 0.392546239155 | 0.407781334059 | rank-2 ${}_2F_1$ walk |

$\delta_{\rm eq}=1/(\mu_0-1)$ is a comparison scale, not the finite-depth exponent.